# HAWK: Honed Anticipation of Wideout Kinematics 🦅

## Starter Notebook - Data Exploration

This notebook explores the NFL Big Data Bowl 2026 Analytics competition data to develop the HAWK Index for measuring wide receiver performance.

**HAWK Index Components:**
- Reaction latency (time from ball release to first movement)
- Acceleration efficiency (rate of speed change)
- Pursuit geometry (optimal path to ball landing point)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("🦅 HAWK Analysis Environment Ready!")
print("📊 Libraries imported successfully")


In [ ]:
# Load the data
data_path = Path("../data/114239_nfl_competition_files_published_analytics_final")

# Load tracking data (Week 1 as example)
tracking = pd.read_csv(data_path / "train/input_2023_w01.csv")

# Load supplementary data (play information)
plays = pd.read_csv(data_path / "supplementary_data.csv")

print(f"📈 Tracking data shape: {tracking.shape}")
print(f"📋 Plays data shape: {plays.shape}")
print(f"\n🎯 Tracking columns: {list(tracking.columns)}")
print(f"\n📝 Plays columns: {list(plays.columns)}")


In [ ]:
# Quick join on game_id and play_id
merged = tracking.merge(plays, on=['game_id', 'play_id'], how='left')

print(f"🔗 Merged data shape: {merged.shape}")
print(f"\n📊 First few rows:")
merged.head()


In [ ]:
# Explore the data structure
print("🔍 DATA EXPLORATION")
print("=" * 50)

print(f"\n📊 Tracking Data Info:")
print(f"- Total frames: {tracking.shape[0]:,}")
print(f"- Unique games: {tracking['game_id'].nunique()}")
print(f"- Unique plays: {tracking['play_id'].nunique()}")
print(f"- Unique players: {tracking['nfl_id'].nunique()}")

print(f"\n👥 Player Positions:")
print(tracking['player_position'].value_counts())

print(f"\n⚡ Player Roles:")
print(tracking['player_role'].value_counts())


In [ ]:
# Focus on wide receivers for HAWK analysis
wr_data = tracking[tracking['player_position'].isin(['WR'])]

print(f"🏃‍♂️ Wide Receiver Data:")
print(f"- WR frames: {wr_data.shape[0]:,}")
print(f"- Unique WRs: {wr_data['nfl_id'].nunique()}")
print(f"- WR plays: {wr_data['play_id'].nunique()}")

# Look at one WR play as example
sample_play = wr_data[wr_data['play_id'] == wr_data['play_id'].iloc[0]]
print(f"\n📋 Sample WR Play:")
print(f"- Frames in play: {len(sample_play)}")
print(f"- Player: {sample_play['player_name'].iloc[0]}")
print(f"- Position: {sample_play['player_position'].iloc[0]}")
print(f"- Ball landing point: ({sample_play['ball_land_x'].iloc[0]:.2f}, {sample_play['ball_land_y'].iloc[0]:.2f})")

sample_play[['frame_id', 'x', 'y', 's', 'a', 'dir']].head(10)


## Ball Release Detection 🎯

To calculate reaction latency, we first need to identify when the ball is released.

**The Formula:**
```
Ball Release Frame = max(input_frame_id) - num_frames_output
```

**Why this works:**
- Input data contains frames from snap through ball release + reaction period
- `num_frames_output` tells us how many frames of reaction data we have AFTER ball release
- So ball release occurs `num_frames_output` frames before the end of input data

**Example:**
- Input frames: 1 to 26
- num_frames_output: 21
- Ball release: Frame 26 - 21 = **Frame 5**
- Post-release reaction data: Frames 6-26 (21 frames) ✅
- Future to predict: 21 frames in output file


In [ ]:
# Calculate ball release frames for all plays
def get_ball_release_frames(tracking_data):
    """
    Calculate ball release frame for each play.
    
    Formula: ball_release_frame = max(input_frame_id) - num_frames_output
    """
    play_info = tracking_data.groupby('play_id').agg({
        'frame_id': 'max',
        'num_frames_output': 'first'
    }).reset_index()
    
    play_info.columns = ['play_id', 'max_input_frame', 'num_output_frames']
    play_info['ball_release_frame'] = (
        play_info['max_input_frame'] - play_info['num_output_frames']
    )
    
    # Filter out invalid calculations
    play_info = play_info[play_info['ball_release_frame'] >= 1]
    
    return play_info

# Calculate for all plays
ball_release_info = get_ball_release_frames(tracking)

print("🎯 Ball Release Frame Calculation")
print("=" * 60)
print(f"Total plays analyzed: {len(ball_release_info)}")
print(f"\nSample results:")
print(ball_release_info.head(10))

# Merge with tracking data
tracking = tracking.merge(
    ball_release_info[['play_id', 'ball_release_frame']], 
    on='play_id', 
    how='left'
)

# For the sample play
sample_play = tracking[tracking['play_id'] == tracking['play_id'].iloc[0]]
print(f"\n📊 Sample Play Analysis:")
print(f"   Play ID: {sample_play['play_id'].iloc[0]}")
print(f"   Input frames: 1 to {sample_play['frame_id'].max()}")
print(f"   Ball release frame: {sample_play['ball_release_frame'].iloc[0]:.0f}")
print(f"   Pre-release frames: 1 to {sample_play['ball_release_frame'].iloc[0]:.0f}")
print(f"   Post-release reaction frames: {sample_play['ball_release_frame'].iloc[0]:.0f + 1} to {sample_play['frame_id'].max()}")
print(f"   ✅ We have {sample_play['frame_id'].max() - sample_play['ball_release_frame'].iloc[0]:.0f} frames of reaction data!")


In [ ]:
# Visualize WR movement for a sample play
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: WR trajectory
axes[0].plot(sample_play['x'], sample_play['y'], 'o-', markersize=4, linewidth=2, label='WR Path')
axes[0].scatter(sample_play['ball_land_x'].iloc[0], sample_play['ball_land_y'].iloc[0], 
                color='red', s=100, marker='*', label='Ball Landing', zorder=5)
axes[0].set_xlabel('X Position (yards)')
axes[0].set_ylabel('Y Position (yards)')
axes[0].set_title(f'WR Trajectory: {sample_play["player_name"].iloc[0]}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Speed and Acceleration over time
axes[1].plot(sample_play['frame_id'], sample_play['s'], 'b-', linewidth=2, label='Speed (yd/s)')
axes[1].plot(sample_play['frame_id'], sample_play['a'], 'r-', linewidth=2, label='Acceleration (yd/s²)')
axes[1].set_xlabel('Frame ID')
axes[1].set_ylabel('Value')
axes[1].set_title('Speed and Acceleration Profile')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🎯 Next Steps for HAWK Development:")
print("1. Identify ball release frame")
print("2. Calculate reaction latency")
print("3. Measure acceleration efficiency")
print("4. Compute pursuit geometry optimization")


## Next Steps

This starter notebook has loaded and explored the basic data structure. Next, we'll develop the HAWK Index components:

1. **Reaction Latency**: Time from ball release to first significant movement
2. **Acceleration Efficiency**: Rate of speed change during pursuit
3. **Pursuit Geometry**: Optimal path analysis to ball landing point

The HAWK Index will combine these metrics to create a comprehensive measure of wide receiver anticipation and movement efficiency.
